In [1]:
# Import required packages
using DifferentialEquations     # Main ODE solver framework
using LinearAlgebra             # For matrix operations (mul!)
using SparseArrays              # Sparse matrix handling
using KLU                       # Sparse LU factorization for CVODE_BDF
using SparseDiffTools           # For matrix_colors and ForwardColorJacCache
using PreallocationTools        # For DiffCache
using SciMLSensitivity, Enzyme
using ADTypes

using Plots                     # For plotting results
using YAML                      # For parsing mechanism files

using Graphs                    # For network graph structures
using GraphPlot                 # For visualizing graphs

# Include the chemistry module
include("Chemistry.jl")

"""
Holds the essential, pre-processed chemical data needed for the hot loop.
Note the removal of redundant lists.
"""
struct ChemistryParameters
    S::SparseMatrixCSC{Float64, Int64}
    kinetics_list::SplitKinetics  # CRITICAL: This must use our abstract type
    species_list::Vector{Species}             # Needed for thermodynamic lookups
end

"""
Holds pre-allocated cache arrays for AD-compatibility and performance.
This structure is perfectly designed!
"""
struct CombustionCache{T}
    r_cache::DiffCache{Vector{T}, Vector{T}}
    dC_dt_cache::DiffCache{Vector{T}, Vector{T}}
    c_p_node_temp_cache::DiffCache{Vector{T}, Vector{T}}
    cp_species_vec_cache::DiffCache{Vector{T}, Vector{T}}
    diffusion_term_cache::DiffCache{Vector{T}, Vector{T}}
    diffusion_term_species_cache::DiffCache{Matrix{T}, Vector{T}}
end
# Note: I added the second type parameter to DiffCache, which is common.
# e.g., DiffCache{Vector{Float64}, Vector{Float64}}

"""
The main parameter object passed to the ODE solver.
This structure is also perfectly designed!
"""
struct CombustionParameters{T}
    chem::ChemistryParameters
    A::SparseMatrixCSC{Float64, Int64} # Discretization matrix (e.g., for diffusion)
    V::Vector{Float64}               # Cell volumes
    cache::CombustionCache{T}
end

CombustionParameters

In [2]:
# Loading the reaction mechanism

# Constants and Configuration
const R = 1.987  # Gas constant (cal/mol·K)

# This map contains three mechanisms: 
# - GRI-Mech 3.0 with 53 species and 325 reactions, filename 'GRI30.yaml'
# - Tianfeng Lu's reduced mechanism for methane, 30 species and 184 reactions, filename 'LuMechanism.yaml'
# - Charlélie Laurent reduced mechanism for methane, 17 species and 82 reactions, filename 'LaurentMechanism.yaml'
# - A 2 step mechanism by  Franzelli et al., 6 species and 2 reactions, filename '2step.yaml'
# - A 1 step mechanism (unrecommended), 6 species and 1 reaction, filename '1step.yaml'

const FILENAME = "GRI30.yaml"

# Load the mechanism file
mechanism_data = load_mechanism_data(FILENAME)

# Process the chemical data
species_list, reaction_list, species_index_map = process_chemical_data(mechanism_data)

# Number of species and reactions
const n_species = length(species_list)
const n_vars = n_species + 1
const n_reactions = length(reaction_list)

# Build the stoichiometric matrix and kinetics list
S = sparse(build_stoichiometric_matrix(reaction_list, n_species))
kinetics_list = build_kinetics_list(reaction_list)

# Create chemistry parameters
chem_params = ChemistryParameters(S, kinetics_list, species_list)

# Creating the reactor network

# Build network graph
const n_nodes = 10
g = SimpleGraph(n_nodes)

# Define network topology
edges = [
    (1, 2), (2, 3), (2, 4), (3, 4), (4, 5),
    (5, 6), (5, 7), (6, 7), (7, 8),
    (3, 9), (6, 10)
]

for (i, j) in edges
    add_edge!(g, i, j)
end

# Create connectivity matrix (negative Laplacian for diffusion formulation)
A = -laplacian_matrix(g)

# Initialize node volumes (can be customized later)
V = ones(Float64, n_nodes)

# Set up the initial state
U0 = zeros(Float64, n_nodes, n_species + 1)

# Define initial conditions for different node types
U0 = zeros(Float64, n_nodes, n_species + 1)
initial_conditions = Dict(
    1 => ChemistryConfig(temperature = 900.0, pressure = 1e5, fuel_mixture = Dict("CH4" => 1/10.57, "H2"  => 0.00), air_percentage = 9.57/10.57),
    10 => ChemistryConfig(temperature = 900.0, pressure = 1e5, fuel_mixture = Dict("CH4" => 0.00, "H2"  => 2/6.76), air_percentage = 4.75/6.76),
    :air => ChemistryConfig(temperature = 900.0, pressure = 1e5, fuel_mixture = Dict("CH4" => 0.00, "H2"  => 0.00), air_percentage = 1.00)
)

for node in 1:n_nodes
    config = get(initial_conditions, node, initial_conditions[:air])
    U0[node, :] = initialize_concentrations(config, species_list, species_index_map)
end

In [3]:
# Define diffusion coefficients
const D_TEMP = 10.0e-6
const D_SPECIES = 1.0e-6

# === Reaction ODE per node - with dot product optimizations ===

# Reaction ODE per node.
function reaction_ode_inplace!(
    du::AbstractVector{<:Real},
    X::AbstractVector{<:Real},
    param::ChemistryParameters,
    r::AbstractVector{<:Real},
    dC_dt::AbstractVector{<:Real},
    cp_species_vec::AbstractVector{<:Real}
)
    T_val = X[1]
    
    # --- 1. Compute Reaction Rates ---
    compute_reaction_rates!(r, X, param.kinetics_list, param.species_list)

    # --- 2. Compute Species Source Terms (dC/dt) ---
    # This uses the pre-computed stoichiometric matrix S.
    mul!(dC_dt, param.S, r)
    @. du[2:end] = dC_dt

    # --- 3. Compute Temperature Change (dT/dt) ---
    concentrations = @view X[2:end]

    # Optimized volumetric heat capacity (unchanged, this was already good)
    # Note: R is your gas constant, assumed defined.
    @inbounds for i in eachindex(concentrations)
        # species_cp() is your function for species heat capacity, assumed to exist.
        cp_species_vec[i] = species_cp(T_val, param.species_list[i].thermo) - R
    end
    c_p_vol = dot(concentrations, cp_species_vec)

    # Reuse cp_species_vec as a vector of entalphies of species
    @inbounds for i in eachindex(concentrations)
        # species_cp() is your function for species heat capacity, assumed to exist.
        cp_species_vec[i] = h0(T_val, param.species_list[i].thermo)
    end
     
    total_heat = dot(cp_species_vec, dC_dt)
    
    # Avoid division by zero if c_p_vol is very small
    du[1] = -total_heat / (c_p_vol + 1e-20)

    # Return the volumetric heat capacity for the diffusion term calculation
    return c_p_vol
end

# Main system ODE - Almost no changes needed here!
function system_ode!(du::AbstractVector{<:Real}, u::AbstractVector{<:Real}, p, t::Real)
    # Unpack parameters and caches (this part is perfect)
    chem, A, V = p.chem, p.A, p.V
    cache = p.cache
    r = get_tmp(cache.r_cache, u)
    dC_dt = get_tmp(cache.dC_dt_cache, u)
    c_p_node_temp = get_tmp(cache.c_p_node_temp_cache, u)
    cp_species_vec = get_tmp(cache.cp_species_vec_cache, u)

    n_vars = size(u, 1) ÷ n_nodes # Safer way to get n_vars
    U = reshape(u, n_nodes, n_vars)
    dU = reshape(du, n_nodes, n_vars)
    fill!(dU, zero(eltype(u)))

    # === 1. Reaction calculations (per-node) ===
    @inbounds for i in 1:n_nodes
        X_node = @view U[i, :]
        du_node = @view dU[i, :]
        X_node .= max.(X_node, 1e-100)
        
        # The call is now cleaner, passing only the necessary `chem` parameter object
        c_p_node_temp[i] = reaction_ode_inplace!(du_node, X_node, chem, r, dC_dt, cp_species_vec)
    end

    # === 2. Diffusion calculations (vectorized) ===
    # This entire section was already perfectly optimized and needs no changes.
    invV = 1.0 ./ V
    T_vec = @view U[:, 1]
    diffusion_term_temp = get_tmp(cache.diffusion_term_cache, u)
    mul!(diffusion_term_temp, A, T_vec)
    @. dU[:, 1] += (D_TEMP * invV / c_p_node_temp) * diffusion_term_temp
    
    C_matrix = @view U[:, 2:end]
    diffusion_term_species = get_tmp(cache.diffusion_term_species_cache, u)
    mul!(diffusion_term_species, A, C_matrix)
    @. dU[:, 2:end] += D_SPECIES .* invV .* diffusion_term_species

    return nothing
end

system_ode! (generic function with 1 method)

In [4]:
# === Jacobian Sparsity and AD Setup ===
function build_local_jacobian_sparsity(param::ChemistryParameters)
    stoich_mat = param.S
    J = spzeros(Bool, n_vars, n_vars)
    for r in 1:size(stoich_mat, 2)
        participants = findall(!iszero, stoich_mat[:, r])
        for i in participants, j in participants
            J[i+1, j+1] = true
        end
    end
    for spec in 1:n_species
        J[spec+1, 1] = true
        J[1, spec+1] = true
    end
    J[1, 1] = true
    return J
end

function build_network_jacobian_sparsity(param::ChemistryParameters, laplacian::SparseMatrixCSC)
    local_jac = build_local_jacobian_sparsity(param)
    n_total = n_vars * n_nodes
    J_network = spzeros(Bool, n_total, n_total)
    for node in 1:n_nodes
        idx_start = (node - 1) * n_vars + 1
        idx_end = node * n_vars
        idx_range = idx_start:idx_end
        J_network[idx_range, idx_range] = local_jac
    end
    lap_rows, lap_cols, _ = findnz(laplacian)
    for (i, j) in zip(lap_rows, lap_cols)
        if i != j
            for var in 1:n_vars
                row_idx = (i - 1) * n_vars + var
                col_idx = (j - 1) * n_vars + var
                J_network[row_idx, col_idx] = true
            end
        end
    end
    return J_network
end

jac_sparsity = build_network_jacobian_sparsity(chem_params, A)
colorvec = matrix_colors(jac_sparsity)

# Determine the chunk size
chunk_size = maximum(colorvec)

65

In [5]:
const PROGRESS_PRINT_ITER_INTERVAL = 1000

# This Ref will hold the next iteration at which we want to print.
# It will be initialized correctly at the start of each solve.
const next_print_iter_ref = Ref(0) # Changed to integer type for iterations

# 1. Condition function: Checks if the current iteration has reached the next scheduled print iteration
function condition_iter_progress(u, t, integrator)
    # Trigger if current iteration is >= next scheduled iteration
    return integrator.iter >= next_print_iter_ref[]
end

# 2. Affect function: Executes when the condition is true
function affect_iter_progress!(integrator)
    # Update the next scheduled print iteration.
    # Ensures we print at the desired interval, even if the solver does more than 1 iter between triggers.
    next_print_iter_ref[] = floor(Int, integrator.iter / PROGRESS_PRINT_ITER_INTERVAL + 1) * PROGRESS_PRINT_ITER_INTERVAL

    # Extract relevant info from the integrator
    u = integrator.u
    A = integrator.p.A # Access your system object from the problem's parameters
    U = reshape(u, n_nodes, n_vars) # Reshape for easier access

    # Get values and format simply
    current_iter = integrator.iter # The current iteration count
    t_ns = integrator.t    # Current time in nanoseconds
    temps_10, temps_1 = U[10, 1] , U[1,1]           # Assuming Temperature is the first variable (index 1) in each node
    CH4_conc = U[1,species_index_map["CH4"] + 1]          # Hydrogen concentration in node 10
    H2_conc = U[10,species_index_map["H2"] + 1]          # Hydrogen concentration in node 10
    dt_ps = integrator.dt  # Timestep in picoseconds

    println("Iter= $(current_iter), t= $(round(t_ns, digits=14)) s, dt = $(round(dt_ps, digits=18)) s")
    println("In node 1 : T= $(round(temps_1, digits=6)) K, [CH4]= $(round(CH4_conc, digits=8)) mol/m3")
    println("In node 10: T= $(round(temps_10, digits=6)) K, [H2] = $(round(H2_conc, digits=8)) mol/m3")

    # Optional warning for very small timesteps
    if abs(integrator.dt) < 1e-21
        println("  WARNING: Extremely small timestep (< 1e-21 s) detected.")
    end
end

# 3. Initialization function: Sets up next_print_iter_ref at the start of each solve
function initialize_iter_progress!(cb, u, t, integrator)
    # Start printing from the first multiple of PROGRESS_PRINT_ITER_INTERVAL
    # after the initial iteration (which is 1)
    next_print_iter_ref[] = 1
end

# Create the progress callback instance
progress_callback = DiscreteCallback(
    condition_iter_progress,
    affect_iter_progress!,
    initialize=initialize_iter_progress!,
    save_positions=(false, false) # Don't save positions from this callback
)
# --- Stiffness Detection Callback ---

function stiffness_condition(u, t, integrator)
    # Based on your observed timestep patterns
    dt_critical = 1e-9   # When you typically need stiff methods
    dt_warning = 1e-8    # Early warning threshold
    
    # Simple timestep-based detection
    if integrator.dt < dt_critical
        return true
    end
    
    # Check for rapid timestep decrease (another stiffness indicator)
    if haskey(integrator.opts.callback.affect!.cache, :prev_dt)
        prev_dt = integrator.opts.callback.affect!.cache[:prev_dt]
        dt_ratio = integrator.dt / prev_dt
        
        if dt_ratio < 0.1 && integrator.dt < dt_warning
            return true
        end
    end
    
    # Store current dt for next iteration
    integrator.opts.callback.affect!.cache[:prev_dt] = integrator.dt
    
    return false
end

# Create terminating callback
stiffness_callback = DiscreteCallback(
    stiffness_condition,
    terminate!  # This ends the integration
)

cb = CallbackSet(progress_callback, stiffness_callback)

CallbackSet{Tuple{}, Tuple{DiscreteCallback{typeof(condition_iter_progress), typeof(affect_iter_progress!), typeof(initialize_iter_progress!), typeof(SciMLBase.FINALIZE_DEFAULT), Nothing}, DiscreteCallback{typeof(stiffness_condition), typeof(terminate!), typeof(SciMLBase.INITIALIZE_DEFAULT), typeof(SciMLBase.FINALIZE_DEFAULT), Nothing}}}((), (DiscreteCallback{typeof(condition_iter_progress), typeof(affect_iter_progress!), typeof(initialize_iter_progress!), typeof(SciMLBase.FINALIZE_DEFAULT), Nothing}(Main.condition_iter_progress, Main.affect_iter_progress!, Main.initialize_iter_progress!, SciMLBase.FINALIZE_DEFAULT, Bool[0, 0], nothing), DiscreteCallback{typeof(stiffness_condition), typeof(terminate!), typeof(SciMLBase.INITIALIZE_DEFAULT), typeof(SciMLBase.FINALIZE_DEFAULT), Nothing}(Main.stiffness_condition, SciMLBase.terminate!, SciMLBase.INITIALIZE_DEFAULT, SciMLBase.FINALIZE_DEFAULT, Bool[1, 1], nothing)))

In [6]:
# === System Initialization ===
u0 = vec(U0)

# Create the parameter object `p` as a NamedTuple

# Function to build the cache
function build_cache(T::Type, n_species, n_reactions, n_nodes, chunk_size)
    # The chunk size is passed to the DiffCache constructor.
    # The second type parameter (e.g., Vector{T}) is what get_tmp will use.
    return CombustionCache{T}(
        DiffCache(zeros(T, n_reactions), chunk_size),          # r_cache
        DiffCache(zeros(T, n_species), chunk_size),            # dC_dt_cache
        DiffCache(zeros(T, n_nodes), chunk_size),              # c_p_node_temp_cache
        DiffCache(zeros(T, n_species), chunk_size),            # cp_species_vec_cache
        DiffCache(zeros(T, n_nodes), chunk_size),              # diffusion_term_cache
        DiffCache(zeros(T, n_nodes, n_species), chunk_size)    # diffusion_term_species_cache
    )
end

p = CombustionParameters(
    chem_params,
    float.(A),
    V,
    build_cache(Float64, n_species, n_reactions, n_nodes, chunk_size)
)

ode_func = ODEFunction(system_ode!, sparsity=jac_sparsity, colorvec=colorvec)

(::ODEFunction{true, SciMLBase.FullSpecialize, typeof(system_ode!), UniformScaling{Bool}, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, SparseMatrixCSC{Bool, Int64}, Nothing, Nothing, Nothing, Nothing, typeof(SciMLBase.DEFAULT_OBSERVED), Vector{Int64}, Nothing, Nothing, Nothing}) (generic function with 1 method)

In [7]:
# --- Pre-Hydrogen Explosion Stiffness ---

tspan_pre_h2 = (0.0, 1e-3) # From 0 to 1 millisecond, or where the ship strands (stiffness detection triggers)

prob_pre_h2 = ODEProblem(ode_func, u0, tspan_pre_h2, p)

algo = ImplicitEuler(
        linsolve=KLUFactorization(),
        autodiff=AutoEnzyme(; function_annotation=Enzyme.Duplicated),
        concrete_jac=true,
        chunk_size=Val{4}(),
        standardtag=Val{true}()
) # Run the faster option untill to high stiffness triggers

sensealg = InterpolatingAdjoint(
    autojacvec = EnzymeVJP(),
    autodiff = AutoEnzyme(; function_annotation=Enzyme.Duplicated),
    checkpointing = true
)

println("\nStarting Phase 1a (Pre-Hydrogen Explosion) ODE solve with $algo...")
println("  Time span: $(tspan_pre_h2)")

@time sol_pre_h2 = solve(prob_pre_h2, algo;
    abstol = 1e-6,  # Tight absolute tolerance
    reltol = 1e-8,   # Tight relative tolerance  
    dtmax = 1e-9,    # Max step size (1 ns)
    dtmin = 1e-16,   # Min step size (0.01 fs)
    dt = 1e-9,       # Initial step size
    qmin = 0.2,      # Conservative step reduction
    qmax = 1.5,      # Conservative step increase
    maxiters = 1e8,   # Allow many iterations
    sensealg=sensealg,
    callback=cb
    
)

println("ODE solve ended; 1st stiffness reached. Return code: $(sol_pre_h2.retcode)")


Starting Phase 1a (Pre-Hydrogen Explosion) ODE solve with ImplicitEuler{4, AutoEnzyme{Nothing, Duplicated}, KLUFactorization, NLNewton{Rational{Int64}, Rational{Int64}, Rational{Int64}, Nothing}, typeof(OrdinaryDiffEqCore.DEFAULT_PRECS), Val{:forward}(), true, true, typeof(OrdinaryDiffEqCore.trivial_limiter!)}(KLUFactorization(true, true), NLNewton{Rational{Int64}, Rational{Int64}, Rational{Int64}, Nothing}(1//100, 10, 1//5, 1//5, false, true, nothing), OrdinaryDiffEqCore.DEFAULT_PRECS, :constant, :PI, OrdinaryDiffEqCore.trivial_limiter!, AutoEnzyme(function_annotation=Duplicated))...
  Time span: (0.0, 0.001)
Iter= 15, t= 6.0e-14 s, dt = 6.1035e-14 s
In node 1 : T= 900.0 K, [CH4]= 1.26194221 mol/m3
In node 10: T= 900.0 K, [H2] = 3.95385098 mol/m3
124.791254 seconds (169.08 M allocations: 8.659 GiB, 3.27% gc time, 99.78% compilation time)
ODE solve ended; 1st stiffness reached. Return code: Terminated


retcode: Terminated
Interpolation: 3rd order Hermite
t: 5-element Vector{Float64}:
 0.0
 0.0
 6.103515625e-14
 6.103515625e-14
 6.103515625e-14
u: 5-element Vector{Vector{Float64}}:
 [900.0, 900.0, 900.0, 900.0, 900.0, 900.0, 900.0, 900.0, 900.0, 900.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [900.0, 900.0, 900.0, 900.0, 900.0, 900.0, 900.0, 900.0, 900.0, 900.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [900.0000000000032, 899.9999999999651, 900.0000000000001, 899.9999999996847, 900.0000000014885, 900.0000000000015, 899.9999999998782, 899.9999999999816, 899.9999999999976, 900.0000000001668  …  1.0e-100, 2.0501604714538585e-75, 2.3365869026236316e-56, 1.5242496294017986e-75, 2.2870006478407573e-91, 1.0e-100, 1.0e-100, 1.0e-100, 7.0352220419597235e-93, 1.0008935416995994e-100]
 [900.0000000000032, 899.9999999999651, 900.0000000000001, 899.9999999996847, 900.0000000014885, 900.0000000000015, 899.9999999998782, 899.9999999999816, 899.9999999999976, 900.00000000016

In [8]:
# --- Optimized Algorithm Selection for Multi-Scale Stiff System ---
SHARED_LINSOLVE = KLUFactorization()
SHARED_AD = AutoEnzyme(; function_annotation=Enzyme.Duplicated)
JACOBIAN_CHUNK = Val(12)  # keep high for large systems

# 1. Not that stiff algorithm - Fast explicit with automatic stiffness detection
alg1 = ImplicitEuler(
    linsolve=SHARED_LINSOLVE,
    autodiff=SHARED_AD,
    concrete_jac=true,
    chunk_size=Val{4}(),
    standardtag=Val{true}()
)
# 2. Moderately stiff (combustion microsecond scale) - L-stable with good accuracy
alg2 = TRBDF2(
    linsolve=SHARED_LINSOLVE,
    autodiff=SHARED_AD,
    concrete_jac=true,
    chunk_size=Val{8}(),
    standardtag=Val{true}(),
    smooth_est=true,
    extrapolant=:linear,
    controller=:PI
)
# 3. Stiff transition regions (approaching explosion event) - High order SDIRK
alg3 = KenCarp47(
    linsolve=SHARED_LINSOLVE,
    autodiff=SHARED_AD,
    concrete_jac=true,
    chunk_size=Val{8}(),
    standardtag=Val{true}(),
    smooth_est=true,
    extrapolant=:linear,
    controller=:PI
)
# 4. Very stiff (explosion dynamics) - Multistep BDF with adaptive order
alg4 = Rodas5(
    linsolve=SHARED_LINSOLVE,
    autodiff=SHARED_AD,
    concrete_jac=true,
    chunk_size=Val{12}(),
    standardtag=Val{true}()
)
# 5. Extremely stiff (reaction speed peak) - High order implicit Runge-Kutta
alg5 = RadauIIA5(
    linsolve=SHARED_LINSOLVE,
    autodiff=SHARED_AD,
    concrete_jac=true,
    chunk_size=Val{12}(),
    standardtag=Val{true}(),
    smooth_est=true,
    extrapolant=:linear,
    controller=:Gustafsson
)  

# Algorithm tuple
algtup = (alg1, alg2, alg3, alg4, alg5) 

# --- Stiffness-Based Choice Function ---
@inline function choice_function(integrator)
    dt = integrator.dt

    # Tunable thresholds — this version biases toward Rosenbrock early
    up_thresh   = [5e-5, 1e-6, 5e-9, 1e-11]
    down_thresh = [1e-5, 5e-7, 2e-9, 5e-12]

    if dt > down_thresh[1]
        return 1  # Rosenbrock23
    elseif dt > down_thresh[2]
        return dt < up_thresh[1] ? 2 : 1  # FBDF2
    elseif dt > down_thresh[3]
        return dt < up_thresh[2] ? 3 : 2  # FBDF4
    elseif dt > down_thresh[4]
        return dt < up_thresh[3] ? 4 : 3  # QNDF
    else
        return dt < up_thresh[4] ? 5 : 4  # QNDF tuned
    end
end


# --- Final Composite Algorithm ---
composite_alg = CompositeAlgorithm(algtup, choice_function)

CompositeAlgorithm(; algs = (ImplicitEuler{4, AutoEnzyme{Nothing, Duplicated}, KLUFactorization, NLNewton{Rational{Int64}, Rational{Int64}, Rational{Int64}, Nothing}, typeof(OrdinaryDiffEqCore.DEFAULT_PRECS), Val{:forward}(), true, true, typeof(OrdinaryDiffEqCore.trivial_limiter!)}(KLUFactorization(true, true), NLNewton{Rational{Int64}, Rational{Int64}, Rational{Int64}, Nothing}(1//100, 10, 1//5, 1//5, false, true, nothing), OrdinaryDiffEqCore.DEFAULT_PRECS, :constant, :PI, OrdinaryDiffEqCore.trivial_limiter!, AutoEnzyme(function_annotation=Duplicated)), TRBDF2{8, AutoEnzyme{Nothing, Duplicated}, KLUFactorization, NLNewton{Rational{Int64}, Rational{Int64}, Rational{Int64}, Nothing}, typeof(OrdinaryDiffEqCore.DEFAULT_PRECS), Val{:forward}(), true, true, typeof(OrdinaryDiffEqCore.trivial_limiter!)}(KLUFactorization(true, true), NLNewton{Rational{Int64}, Rational{Int64}, Rational{Int64}, Nothing}(1//100, 10, 1//5, 1//5, false, true, nothing), OrdinaryDiffEqCore.DEFAULT_PRECS, true, :linea

In [9]:
# --- Non-Stiffness Detection Callback ---

function nonstiffness_condition(u, t, integrator)
    # Detect when stiffness is no longer significant enough to justify stiff methods
    dt_threshold = 1e-12   # Larger than for stiffness; adjust to match your method
    norm_threshold = 1e6   # Lower thresholds than stiffness detection

    # Get current derivative
    f = get_tmp_cache(integrator)[1]
    integrator.f(f, u, integrator.p, t)

    u_norm = norm(u)
    f_norm = norm(f)

    # Heuristics for non-stiff behavior
    large_timestep = integrator.dt > dt_threshold
    small_derivatives = (u_norm < norm_threshold || f_norm < norm_threshold) || (f_norm/u_norm < norm_threshold / 1e2)

    # Return true to trigger callback (terminate integration)
    return large_timestep && small_derivatives
end

# Create terminating callback for non-stiffness
nonstiffness_callback = DiscreteCallback(
    nonstiffness_condition,
    terminate!  # Terminates the integration when no longer stiff
)

cb = CallbackSet(progress_callback, nonstiffness_callback)

CallbackSet{Tuple{}, Tuple{DiscreteCallback{typeof(condition_iter_progress), typeof(affect_iter_progress!), typeof(initialize_iter_progress!), typeof(SciMLBase.FINALIZE_DEFAULT), Nothing}, DiscreteCallback{typeof(nonstiffness_condition), typeof(terminate!), typeof(SciMLBase.INITIALIZE_DEFAULT), typeof(SciMLBase.FINALIZE_DEFAULT), Nothing}}}((), (DiscreteCallback{typeof(condition_iter_progress), typeof(affect_iter_progress!), typeof(initialize_iter_progress!), typeof(SciMLBase.FINALIZE_DEFAULT), Nothing}(Main.condition_iter_progress, Main.affect_iter_progress!, Main.initialize_iter_progress!, SciMLBase.FINALIZE_DEFAULT, Bool[0, 0], nothing), DiscreteCallback{typeof(nonstiffness_condition), typeof(terminate!), typeof(SciMLBase.INITIALIZE_DEFAULT), typeof(SciMLBase.FINALIZE_DEFAULT), Nothing}(Main.nonstiffness_condition, SciMLBase.terminate!, SciMLBase.INITIALIZE_DEFAULT, SciMLBase.FINALIZE_DEFAULT, Bool[1, 1], nothing)))

In [ ]:
# --- Stiff Hydrogen Explosion Timescale ---
tspan_h2_explosion = (sol_pre_h2.t[end], 1e-3) # Until 1 microseconds or when stiffness is dropping
u0_h2_explosion = sol_pre_h2[end] 

prob_h2_explosion = ODEProblem(ode_func, u0_h2_explosion, tspan_h2_explosion, p)

algo = composite_alg

println("\nStarting Phase 1b (Hydrogen Explosion) ODE solve with composite algorithm...")
println("  Time span: $(tspan_h2_explosion)")

@time sol_h2_explosion = solve(prob_h2_explosion, algo;
    abstol = 1e-4,  # Tight absolute tolerance
    reltol = 1e-6,   # Tight relative tolerance  
    dtmax = 1e-9,    # Max step size (1 ns)
    dtmin = 1e-32,   # Min step size (0.01 fs)
    dt = 1e-9,       # Initial step size
    qmin = 0.2,      # Conservative step reduction
    qmax = 1.5,      # Conservative step increase
    maxiters = 1e8,   # Allow many iterations
    callback=cb,
    sensealg=sensealg
)

println("Phase 1b (Hydrogen Explosion) ODE solve completed. Return code: $(sol_h2_explosion.retcode)")

In [ ]:
# --- Mildly Stiff Interbellum Timescale ---

cb = CallbackSet(progress_callback, stiffness_callback)

tspan_interbellum = (sol_h2_explosion.t[end], 0.010) # until 10 millisecond or stiffness returns
u0_interbellum = sol_h2_explosion[end] 

prob_interbellum = ODEProblem(ode_func, u0_interbellum, tspan_ch4_explosion, p)

algo = AutoTsit5(Rosenbrock23())

println("\nStarting Phase 2a (Interbellum) ODE solve with $algo...")
println("  Time span: $(tspan_interbellum)")

@time sol_interbellum = solve(problem, algo;
                   reltol=1e-8,
                   abstol=1e-10,
                   maxiters=1e9,
                   callback=cb,
                   sensealg=sensealg
)

println("Phase 2a (Interbellum) ODE solve completed. Return code: $(sol_interbellum.retcode)")

In [ ]:
# --- Stiff Methane Explosion Timescale ---

cb = CallbackSet(progress_callback, nonstiffness_callback)

tspan_ch4_explosion = (sol_interbellum.t[end], 0.1) 
u0_ch4 = sol_interbellum[end] 

prob_ch4 = ODEProblem(ode_func, u0_ch4, tspan_ch4_explosion, p)

algo = composite_algorithm

println("\nStarting Phase 2b (Methane Explosion) ODE solve with $algo...")
println("  Time span: $(tspan_ch4_explosion)")

@time sol_ch4 = solve(prob_ch4, algo;
                   reltol=1e-8,
                   abstol=1e-10,
                   maxiters=1e9,
                   callback=cb,
                   sensealg=sensealg
)
println("Phase 2b (Methane Explosion) ODE solve completed. Return code: $(sol_ch4.retcode)")

In [ ]:
# --- Plotting ---

# Merging the solutions
sol_initial = SciMLBase.merge(SciMLBase.merge(sol_pre_h2, sol_h2_explosion),SciMLBase.merge(sol_interbellum, sol_ch4))

function plot_time_evolution(sol, nodes, var_id, n_nodes, n_vars)
    time_series = zeros(length(sol.t), length(nodes))
    
    for (i, u_vec) in enumerate(sol.u)
        U = reshape(u_vec, n_nodes, n_vars)
        time_series[i, :] = U[nodes, var_id]
    end
    
    return plot(sol.t, time_series, label=["Node $n" for n in nodes'])
end

# Plotting functions (kept for completeness, but also can be moved)
ch4_idx = species_index_map["CH4"] + 1
h2_idx = species_index_map["H2"] + 1
o2_idx = species_index_map["O2"] + 1

# Plot 1: Temperature evolution
p1 = plot_time_evolution(sol_initial, [1, 2, 4, 5, 6, 10], 1, n_nodes, n_variables)
xlabel!("Time (s)")
ylabel!("Temperature (K)")
savefig("network_temperature.png")
display(p1)

# Plot 2: CH₄ concentration
p2 = plot_time_evolution(sol_initial, [1, 2, 4], ch4_idx, n_nodes, n_variables)
xlabel!("Time (s)")
ylabel!("CH₄ Concentration (mol/m³)")
savefig("network_ch4.png")
display(p2)

# Plot 3: H₂ concentration
p3 = plot_time_evolution(sol_h2, [5, 6, 10], h2_idx, n_nodes, n_variables)
xlabel!("Time (s)")
ylabel!("H₂ Concentration (mol/m³)")
savefig("network_h2.png")
display(p3)

# Plot 4: O₂ concentration
p4 = plot_time_evolution(sol_initial, [1, 2, 4, 5, 6, 10], o2_idx, n_nodes, n_variables)
xlabel!("Time (s)")
ylabel!("O₂ Concentration (mol/m³)")
savefig("network_o2.png")
display(p4)

In [ ]:
# --- 3rd phase of simulation: Diffusion ---

# Extended simulation parameters
tspan_extended = (sol_initial.t[end], 1.0) # Continue from where the first solve left off
u0_extended = sol_initial[end] # Final state from previous simulation

# Create new problem for extended simulation
prob_extended = ODEProblem(ode_func, u0_extended, tspan_extended, p)

algo = AutoVern9(Rodas5(
    linsolve=KLUFactorization(),
    autodiff=AutoEnzyme()))

println("\nStarting extended simulation with QNDF...")
println("  Time span: $(tspan_extended)")

@time prob_extended = solve(problem, algo;
                   reltol=1e-8,
                   abstol=1e-10,
                   maxiters=1e9,
                   callback=cb,
                   sensealg=sensealg
)

println("Extended simulation completed successfully!")

In [ ]:

co_idx = species_index_map["CO"] + 1
h2o_idx = species_index_map["H2O"] + 1
co2_idx = species_index_map["CO2"] + 1

time_points = sol_extended.t

# Plot 1: Temperature evolution
p5 = plot_time_evolution(sol_extended, [1, 2, 4, 5, 6, 10], 1, n_nodes, n_variables)
xlabel!("Time (s)")
ylabel!("Temperature (K)")
savefig("network_temperature_long.png")
display(p5)

# Plot 2: CO concentration
p6 = plot_time_evolution(sol_extended, [1, 2, 4], co_idx, n_nodes, n_variables)
xlabel!("Time (s)")
ylabel!("CO Concentration (mol/m³)")
savefig("network_co.png")
display(p6)

# Plot 3: H₂ concentration
p7 = plot_time_evolution(sol_extended, [5, 6, 10], h2o_idx, n_nodes, n_variables)
xlabel!("Time (s)")
ylabel!("H₂O Concentration (mol/m³)")
savefig("network_h2o.png")
display(p7)

# Plot 4: O₂ concentration
p8 = plot_time_evolution(sol_extended, [1, 2, 4, 5, 6, 10], co2_idx, n_nodes, n_variables)
xlabel!("Time (s)")
ylabel!("CO₂ Concentration (mol/m³)")
savefig("network_co2.png")
display(p8)